# 4. Menghapus Data Duplikat dan Memeriksa Missing Values

Tahap selanjutnya adalah melakukan *data cleaning* terhadap data yang ditemukan pada pemeriksaan sebelumnya.

Data duplikat dihapus karena transaksi yang tercatat lebih dari satu kali dapat menyebabkan hasil perhitungan jumlah transaksi, penjualan, maupun pendapatan menjadi lebih besar dari kondisi sebenarnya.

Setelah data duplikat dihapus, dilakukan pemeriksaan kembali terhadap *missing values* pada kolom yang berhubungan langsung dengan analisis, yaitu:

- `order_id`
- `product_categories`
- `total_qty`
- `Total Pembayaran`
- `order_date`
- `Kota/Kabupaten`

Data yang memiliki `order_date` kosong nantinya tidak digunakan untuk analisis tren tahunan karena tidak dapat ditentukan tahun transaksinya.

Sementara itu, data dengan `total_qty` kosong tidak digunakan ketika menghitung jumlah barang yang dibeli.

In [1]:
import pandas as pd
import numpy as np

# Membaca dataset
df = pd.read_csv("../data/raw/indonesia_ecommerce_sales_challenging.csv", sep=";")

# Menyimpan jumlah data sebelum cleaning
jumlah_awal = len(df)

# Menghapus data duplikat
df = df.drop_duplicates().copy()

# Menampilkan jumlah duplikat yang dihapus
print("Jumlah duplikat yang dihapus:",
      jumlah_awal - len(df))

# Kolom penting untuk analisis
kolom_penting = [
    "order_id",
    "product_categories",
    "total_qty",
    "Total Pembayaran",
    "order_date",
    "Kota/Kabupaten"
]

# Memeriksa missing values
print("\nMissing values pada kolom penting:")
display(
    df[kolom_penting]
    .isnull()
    .sum()
)

Jumlah duplikat yang dihapus: 0

Missing values pada kolom penting:


order_id                0
product_categories      0
total_qty             283
Total Pembayaran        0
order_date            188
Kota/Kabupaten          0
dtype: int64

# 5. Memastikan Tipe Data dan Merapikan Data

Setelah data duplikat diperiksa, tahap berikutnya adalah memastikan tipe data setiap kolom sesuai dengan kebutuhan analisis.

Kolom `order_date` dan `Waktu Pesanan Dibuat` diubah menjadi tipe data **datetime** agar dapat digunakan untuk mengambil tahun transaksi dan melakukan analisis berdasarkan waktu.

Kolom numerik seperti `total_qty`, `Total Pembayaran`, `Total Diskon`, dan `total_returned_qty` dipastikan memiliki tipe data numerik.

Selain itu, data pada kolom wilayah dan status pesanan dirapikan dengan menghapus spasi yang tidak diperlukan dan menyeragamkan huruf menjadi huruf kapital. Hal ini penting agar nilai seperti `KAB. BOGOR` dapat difilter secara konsisten.

Nama kolom juga diseragamkan menjadi huruf kecil dengan format *snake_case* agar lebih mudah digunakan pada tahap analisis.

In [2]:
# Menyeragamkan nama kolom
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[/\s]+", "_", regex=True)
    .str.replace(r"_+", "_", regex=True)
    .str.strip("_")
)

# Mengubah kolom tanggal menjadi datetime
df["order_date"] = pd.to_datetime(
    df["order_date"],
    errors="coerce"
)

df["waktu_pesanan_dibuat"] = pd.to_datetime(
    df["waktu_pesanan_dibuat"],
    errors="coerce"
)

# Mengubah kolom numerik menjadi tipe numerik
kolom_numerik = [
    "total_qty",
    "total_weight_gr",
    "total_returned_qty",
    "total_diskon",
    "num_product_categories",
    "ongkos_kirim_dibayar_oleh_pembeli",
    "estimasi_potongan_biaya_pengiriman",
    "total_pembayaran",
    "perkiraan_ongkos_kirim"
]

for kolom in kolom_numerik:
    if kolom in df.columns:
        df[kolom] = pd.to_numeric(
            df[kolom],
            errors="coerce"
        )

# Membersihkan kolom teks
kolom_teks = [
    "product_categories",
    "status_pesanan",
    "kota_kabupaten",
    "provinsi",
    "metode_pembayaran"
]

for kolom in kolom_teks:
    if kolom in df.columns:
        df[kolom] = (
            df[kolom]
            .astype("string")
            .str.strip()
        )

# Menyeragamkan wilayah dan provinsi
df["kota_kabupaten"] = (
    df["kota_kabupaten"]
    .str.upper()
)

df["provinsi"] = (
    df["provinsi"]
    .str.upper()
)

# Melihat hasil akhir tipe data
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18868 entries, 0 to 18867
Data columns (total 19 columns):
 #   Column                              Non-Null Count  Dtype         
---  ------                              --------------  -----         
 0   order_id                            18868 non-null  str           
 1   total_qty                           18585 non-null  float64       
 2   total_weight_gr                     18868 non-null  int64         
 3   total_returned_qty                  18868 non-null  int64         
 4   total_diskon                        18868 non-null  int64         
 5   product_categories                  18868 non-null  string        
 6   num_product_categories              18868 non-null  int64         
 7   status_pesanan                      18868 non-null  string        
 8   alasan_pembatalan                   18868 non-null  str           
 9   opsi_pengiriman                     18868 non-null  str           
 10  metode_pembayaran                

# 6. Menentukan Transaksi Penjualan yang Valid

Untuk analisis penjualan dan pendapatan, tidak semua transaksi dalam dataset dapat dianggap sebagai penjualan yang berhasil.

Transaksi dengan status `Batal` tidak digunakan karena pesanan tersebut tidak berhasil diselesaikan.

Oleh karena itu, analisis difokuskan pada transaksi dengan status **`Selesai`**. Data transaksi selesai disimpan ke dalam DataFrame baru bernama `df_selesai`, sedangkan dataset utama `df` tetap dipertahankan.

Penggunaan transaksi selesai ini akan diterapkan pada analisis pendapatan, tren penjualan tahunan, serta analisis kategori di Kabupaten Bogor.

In [3]:
# Melihat distribusi status pesanan
print("Distribusi status pesanan:")
display(
    df["status_pesanan"]
    .value_counts(dropna=False)
)

# Mengambil transaksi yang sudah selesai
df_selesai = df[
    df["status_pesanan"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("selesai")
].copy()

print("\nJumlah seluruh data:", len(df))
print("Jumlah transaksi selesai:", len(df_selesai))

Distribusi status pesanan:


status_pesanan
Selesai                                                                                   16045
Batal                                                                                      2573
Sedang Dikirim                                                                               59
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-08.       38
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-06.       34
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-07.       34
Telah Dikirim                                                                                30
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-05.       25
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-09.       22
Pesanan diterima, namun Pembeli masih dapat mengajukan pengembalian hingga 2025-12-10.        6
Pesanan diterima, namun P


Jumlah seluruh data: 18868
Jumlah transaksi selesai: 16045


# 7. Menyiapkan Data untuk Analisis

Pada tahap ini, data yang telah dibersihkan disiapkan untuk menjawab pertanyaan analisis.

### Analisis Tren Penjualan Tahunan

Untuk mengetahui tren penjualan setiap tahun, digunakan transaksi yang telah selesai dan memiliki `order_date` serta `total_pembayaran`. Tahun transaksi diambil dari kolom `order_date`, kemudian total pembayaran dijumlahkan berdasarkan tahun.

### Analisis Kabupaten Bogor

Untuk mengetahui kategori yang paling banyak dibeli di Kabupaten Bogor, data difilter berdasarkan `kota_kabupaten = KAB. BOGOR` dan hanya menggunakan transaksi dengan status `Selesai`.

Kolom `product_categories` kemudian diperiksa karena satu transaksi dapat memiliki lebih dari satu kategori, misalnya:

`Keranjang, Other, Tempat Nasi`

Kategori tersebut perlu dipisahkan berdasarkan tanda koma agar setiap kategori dapat dihitung secara terpisah.

Untuk menentukan kategori yang paling sering muncul, digunakan frekuensi kemunculan kategori dalam transaksi Kabupaten Bogor.

Namun, `total_qty` tidak langsung dijumlahkan untuk setiap kategori pada transaksi gabungan. Hal tersebut dilakukan untuk menghindari kesalahan karena jumlah barang pada transaksi gabungan tidak diketahui secara spesifik untuk masing-masing kategori.

Dengan demikian, hasil analisis kategori digunakan untuk mengetahui **kategori yang paling sering dibeli/muncul dalam transaksi**, sedangkan jumlah unit per kategori hanya dihitung pada transaksi yang memiliki satu kategori.

In [4]:
# =========================================================
# A. MENYIAPKAN DATA UNTUK TREN PENJUALAN TAHUNAN
# =========================================================

df_tren = df_selesai.dropna(
    subset=[
        "order_date",
        "total_pembayaran"
    ]
).copy()

# Mengambil tahun dari tanggal transaksi
df_tren["tahun"] = (
    df_tren["order_date"]
    .dt.year
)

# Menghitung total penjualan setiap tahun
tren_tahunan = (
    df_tren
    .groupby("tahun")
    .agg(
        jumlah_transaksi=("order_id", "nunique"),
        total_penjualan=("total_pembayaran", "sum")
    )
    .reset_index()
    .sort_values("tahun")
)

print("Tren penjualan tahunan:")
display(tren_tahunan)


# =========================================================
# B. MENYIAPKAN DATA KABUPATEN BOGOR
# =========================================================

df_bogor = df_selesai[
    df_selesai["kota_kabupaten"]
    .astype("string")
    .str.strip()
    .str.upper()
    .eq("KAB. BOGOR")
].copy()

print("Jumlah transaksi selesai di Kabupaten Bogor:",
      len(df_bogor))

# Melihat contoh data Kabupaten Bogor
display(
    df_bogor[
        [
            "order_id",
            "product_categories",
            "total_qty",
            "kota_kabupaten"
        ]
    ].head()
)


# =========================================================
# C. MEMERIKSA DATA KATEGORI KABUPATEN BOGOR
# =========================================================

print("Missing values:")
display(
    df_bogor[
        [
            "product_categories",
            "total_qty"
        ]
    ].isnull().sum()
)


# =========================================================
# D. MEMISAHKAN KATEGORI YANG BERGABUNG
# =========================================================

df_bogor_kategori = df_bogor[
    df_bogor["product_categories"]
    .notna()
].copy()

# Memisahkan kategori berdasarkan tanda koma
df_bogor_kategori["kategori"] = (
    df_bogor_kategori["product_categories"]
    .astype("string")
    .str.split(",")
)

# Mengubah satu baris dengan banyak kategori
# menjadi beberapa baris
df_bogor_kategori = (
    df_bogor_kategori
    .explode("kategori")
)

# Membersihkan spasi kategori
df_bogor_kategori["kategori"] = (
    df_bogor_kategori["kategori"]
    .str.strip()
)

# Menghapus kategori kosong
df_bogor_kategori = df_bogor_kategori[
    df_bogor_kategori["kategori"].notna()
    & df_bogor_kategori["kategori"].ne("")
].copy()


# =========================================================
# E. MENENTUKAN KATEGORI YANG PALING SERING DIBELI
# =========================================================

kategori_bogor = (
    df_bogor_kategori
    .groupby("kategori")
    .agg(
        jumlah_transaksi=("order_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "jumlah_transaksi",
        ascending=False
    )
)

print("Kategori yang paling banyak muncul dalam transaksi Kabupaten Bogor:")
display(kategori_bogor.head(10))


# =========================================================
# F. JUMLAH BARANG UNTUK TRANSAKSI SATU KATEGORI
# =========================================================

df_bogor_satu_kategori = df_bogor[
    df_bogor["product_categories"]
    .notna()
    & df_bogor["total_qty"].notna()
    & ~df_bogor["product_categories"]
    .astype("string")
    .str.contains(",", regex=False)
].copy()

jumlah_barang_bogor = (
    df_bogor_satu_kategori
    .groupby("product_categories")
    .agg(
        total_barang=("total_qty", "sum"),
        jumlah_transaksi=("order_id", "nunique")
    )
    .reset_index()
    .sort_values(
        "total_barang",
        ascending=False
    )
)

print("Jumlah barang berdasarkan transaksi satu kategori:")
display(jumlah_barang_bogor.head(10))

Tren penjualan tahunan:


,tahun,jumlah_transaksi,total_penjualan
0,2023,343,23357121
1,2024,7510,478801256
2,2025,7688,424677937


Jumlah transaksi selesai di Kabupaten Bogor: 955


,order_id,product_categories,total_qty,kota_kabupaten
2,ORD_0000003,Celengan,1.0,KAB. BOGOR
7,ORD_0000008,Celengan,1.0,KAB. BOGOR
47,ORD_0000048,Rak / Rak Serbaguna,1.0,KAB. BOGOR
49,ORD_0000050,Celengan,1.0,KAB. BOGOR
60,ORD_0000061,Celengan,1.0,KAB. BOGOR


Missing values:


product_categories     0
total_qty             10
dtype: int64

Kategori yang paling banyak muncul dalam transaksi Kabupaten Bogor:


,kategori,jumlah_transaksi
6,Celengan,350
2,Aksesoris Pintu,170
12,Mangkok Sambal / Saus,123
13,Nampan / Tray,49
4,Baskom / Mangkok Besar,39
29,Tempat Nasi,37
14,Other,34
24,Seal / Baut / Roof,32
10,Lunch Box / Rantang,28
9,Keranjang,27


Jumlah barang berdasarkan transaksi satu kategori:


,product_categories,total_barang,jumlah_transaksi
4,Celengan,407.0,331
8,Lunch Box / Rantang,334.0,26
0,Aksesoris Pintu,208.0,159
9,Mangkok Sambal / Saus,198.0,111
10,Nampan / Tray,102.0,43
21,Seal / Baut / Roof,96.0,32
2,Baskom / Mangkok Besar,69.0,34
25,Tempat Nasi,38.0,34
7,Keranjang,27.0,16
3,Botol / Gelas / Mug,25.0,12


# 8. Menyimpan Dataset Hasil Cleaning

Setelah seluruh proses *data cleaning* dan persiapan analisis selesai, dilakukan validasi akhir untuk memastikan tidak terdapat duplikat dan kolom-kolom penting sudah memiliki tipe data yang sesuai.

Dataset yang telah dibersihkan kemudian disimpan dalam bentuk CSV agar dapat digunakan kembali pada tahap **Exploratory Data Analysis (EDA)** dan visualisasi.

Selain dataset utama, data khusus Kabupaten Bogor dan hasil pemisahan kategori juga disimpan agar dapat digunakan untuk menjawab pertanyaan analisis mengenai kategori yang paling banyak dibeli di wilayah tersebut.

Penyimpanan dataset dilakukan dengan `index=False` agar indeks DataFrame tidak ikut tersimpan sebagai kolom tambahan.

In [5]:
# Validasi akhir
print("Ukuran dataset setelah cleaning:", df.shape)

print("Jumlah duplikat:",
      df.duplicated().sum())

print("\nMissing values kolom utama:")
display(
    df[
        [
            "order_id",
            "product_categories",
            "total_qty",
            "total_pembayaran",
            "order_date",
            "kota_kabupaten"
        ]
    ].isnull().sum()
)


# =========================================================
# MENYIMPAN DATASET
# =========================================================

# Dataset utama hasil cleaning
df.to_csv(
    "../data/clean/indonesia_ecommerce_sales_clean.csv",
    index=False
)

print("\nDataset berhasil disimpan.")

Ukuran dataset setelah cleaning: (18868, 19)
Jumlah duplikat: 0

Missing values kolom utama:


order_id                0
product_categories      0
total_qty             283
total_pembayaran        0
order_date            593
kota_kabupaten          0
dtype: int64


Dataset berhasil disimpan.
